# Proyecto 5 OPCIÓN (A): Topología de Bitcoin y Ethereum
**Diplomado de Modelado y Simulación - Módulo de Grafos y TDA**

**Instructores:** Dr. Jesús F. Espinoza & Dra. Rosalía G. Hernández

---

Esta libreta es la base para el Proyecto 5 (A). Utilizaremos la metodología del artículo *Sliding Windows and Persistence* (Perea & Harer, 2015) para analizar series de tiempo, y usaremos el TDA para analizar la dinámica acoplada de las dos criptomonedas más grandes.

**Objetivo:**
1. **Analizar** la "forma" de una sola serie de tiempo (Bitcoin) usando Ventanas Deslizantes (Takens). Una serie de tiempo puede convertirse en una nube de puntos geométricos. La "forma" de esa nube nos dice si el sistema es estable, periódico o caótico.
2. **(Reto Avanzado)** Analizar la "danza" entre **Bitcoin y Ethereum**. ¿Se mueven juntos? ¿Uno persigue al otro? ¿Cómo cambia su relación en las crisis?

### Estructura de la Libreta

- **Parte 1:** La Herrramienta Matemática (Ventanas Deslizantes)
- **Parte 2:** Cuantificando Estructura con TDA
- **Parte 3:** Bitcoin Individual (Datos Reales)
- **Parte 4:** Parte Experimental e Instrucciones
  - Reto 1. Monitor de Regímenes de Mercado
  - Reto 2. BTC vs ETH

In [ ]:
# 1. Instalación de librerías
!pip install -q ripser persim yfinance

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from ripser import ripser
from persim import plot_diagrams

print("Librerías listas.")

## Parte 1: La Herramienta Matemática (Ventanas Deslizantes)

Para analizar topología, necesitamos puntos en el espacio, no una línea en el tiempo. Usamos la técnica de **Time Delay Embedding** (o Ventanas Deslizantes).

Si tenemos la serie $x(t)$, creamos vectores:
$$ v(t) = [x(t), x(t+\tau), x(t+2\tau), ...] $$

* **Dimensión ($d$):** Cuántos puntos tomamos.
* **Retardo ($\tau$):** Cuántos pasos nos saltamos.

In [ ]:
def sliding_window_embedding(series, dim, tau=1):
    """
    Convierte una serie de tiempo 1D en una nube de puntos d-dimensional.
    """
    N = len(series)
    num_vectors = N - (dim - 1) * tau
    if num_vectors <= 0:
        raise ValueError("La serie es muy corta para esa dimensión y retardo.")

    embedding = np.zeros((num_vectors, dim))
    for i in range(num_vectors):
        # Tomamos 'dim' puntos separados por 'tau' pasos
        embedding[i, :] = series[i : i + dim * tau : tau]

    return embedding

In [ ]:
# --- EJEMPLO VISUAL ---
# Generamos una señal periódica (Coseno)
t = np.linspace(0, 4*np.pi, 100)
senal_simple = np.cos(t)

# La convertimos a 2D (Dimensión 2)
nube_circular = sliding_window_embedding(senal_simple, dim=2, tau=15)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(senal_simple); plt.title("Serie de Tiempo (1D)")
plt.subplot(1, 2, 2)
plt.scatter(nube_circular[:,0], nube_circular[:,1]); plt.title("Embedding (2D)")
plt.show()

print("¡Nota cómo la onda se convierte en un círculo!")

## Parte 2: Cuantificando Estructura con TDA

El artículo de Perea & Harer nos dice que:
1.  Si la señal es **periódica/recurrente**, la nube forma un **círculo** (o ciclo).
2.  Podemos medir qué tan "perfecto" es ese círculo usando la **Persistencia Máxima de $H_1$**.
3.  **Persistencia Alta** = Señal Estructurada/Repetitiva.
4.  **Persistencia Baja** = Ruido o Caos sin estructura.

Vamos a crear una función para calcular este "Score Topológico".

In [ ]:
# Función para calcular el Score (Vida del ciclo más largo)
def calcular_score_topologico(nube_puntos):
    """
    Calcula la persistencia máxima de H1 (ciclos).
    Retorna 0 si no hay ciclos.
    """
    # Ripser calcula diagramas. maxdim=1 nos da H0 y H1.
    # Usamos un threshold infinito para ver todas las estructuras.

    try:
        diagramas = ripser(nube_puntos, maxdim=1)['dgms']
    except:
        return 0 # En caso de error por datos vacíos

    if len(diagramas) < 2: return 0 # No se encontró H1

    h1 = diagramas[1]
    if len(h1) == 0: return 0

    # Vida = Muerte - Nacimiento
    vidas = h1[:, 1] - h1[:, 0]

    # El score es la vida del ciclo más persistente
    return np.max(vidas)

# Probamos el score en nuestro círculo
score = calcular_score_topologico(nube_circular)
print(f"Score Topológico del Coseno: {score:.4f} (Alto = Estructura)")

## Parte 3: Bitcoin Individual (Datos Reales)




Descargamos datos de Bitcoin para verificar que nuestra herramienta funciona.

In [ ]:
# Descargar datos de BITCOIN
ticker = "BTC-USD"
datos = yf.download(ticker, start="2018-01-01", end="2024-01-01")

# Nos quedamos solo con el precio de cierre ajustado
precios = datos['Close'].values.flatten()

plt.figure(figsize=(12, 4))
plt.plot(precios)
plt.title(f"Precio Histórico {ticker}")
plt.xlabel("Días")
plt.ylabel("Precio")
plt.show()

print(f"Tenemos {len(precios)} días de datos.")

Calculemos el score **global** de toda la historia. (Esto puede tardar un poco, si la serie es larga)

In [ ]:
# Normalización simple (Z-score global)
precios_norm = (precios - np.mean(precios)) / np.std(precios)

# Crear nube de puntos (Embedding 3D)
nube_btc = sliding_window_embedding(precios_norm, dim=3, tau=2)

# Calcular Score de toda la historia
score_total = calcular_score_topologico(nube_btc)
print(f"Score Global de Estructura BTC: {score_total:.4f}")

## Parte 4: Los Retos

### EL RETO 1 (Monitor de Regímenes de Mercado)

El mercado financiero no es una onda perfecta, pero tiene "estructuras" (atractores) que se rompen durante las crisis.

**Objetivo:** Construir una serie de tiempo de **Scores Topológicos** para detectar crisis.

**Instrucciones:**
1.  Definan un tamaño de ventana de observación (ej. `ventana_observacion = 100` días).
2.  Recorran la serie de precios `precios` usando un bucle.
3.  En cada paso $t$, tomen el fragmento de precio: `fragmento = precios[t : t + ventana_observacion]`.
4.  Normalicen ese fragmento (importante: resten la media y dividan por la desviación estándar para que la escala del precio no afecte).
5.  Conviertan el fragmento en una nube de puntos (Embedding). *Sugerencia: Prueben `dim=3` o `dim=4` y `tau=1` o `tau=2`*.
6.  Calculen el `score_topologico` de esa nube.
7.  Guarden el score en una lista.
8.  **Resultado Final:** Grafiquen la serie de Scores debajo de la serie de Precios. ¿Qué pasa con el Score durante la crisis de 2008 (aprox día 2000-2500) o el COVID (2020)?

**Hipótesis a verificar:** ¿El mercado se vuelve más "estructurado" (score alto) o más "aleatorio" (score bajo) antes de una crisis?

### EL RETO 2 (BTC vs ETH)

Aquí analizaremos la relación dinámica entre Bitcoin y Ethereum utilizando topología. No solo consiste en calcular una correlación estadística simple (como un número de Pearson) sino una correlación **topológica** o **geométrica**.

En lugar de usar retardos de tiempo, usaremos **dos activos simultáneos** como coordenadas. El estado del mercado en el tiempo $t$ será un punto en el plano:
$$ P(t) = (PrecioBTC_t, PrecioETH_t) $$

**Instrucciones:**
1.  Descarguen precios históricos de `['BTC-USD', 'ETH-USD']`.
2.  **Alineación:** Usen `.dropna()` para eliminar días donde una moneda no existía o el mercado estaba cerrado.
3.  **Normalización:** Normalicen AMBAS monedas independientemente (Reste media, divida por desviación estándar). *Si no hacen esto, Bitcoin (60k USD) aplastará a Ethereum (3k USD) en el gráfico.*
4. Grafiquen un Scatter Plot: Eje X = BTC, Eje Y = ETH.
  * ¿Es una línea recta diagonal? Si ven esto significa **Correlación perfecta** (se mueven idénticos). Topológicamente es simple (dimensión 1)
  * ¿Hay bucles/ciclos? Si ven que la trayectoria forma ``ciclos'' abiertos, significa que **hay un desfase** (uno sube primero y el otro lo sigue después). Esto es estructura topológica ($H_1$).
  * ¿Es una nube dispersa? Significa que no hay relación clara (desorden).

5.  Definan una ventana de tiempo (ej. 60 días).
6.  Recorran la historia tomando fragmentos de 60 días de su trayectoria 2D y calculen la **Persistencia Máxima ($H_1$)** para cada ventana.
7. **Resultado final:** Grafiquen la serie de tiempo de los Scores Topológicos (eje Y izquierdo) junto con el Precio de Bitcoin (eje Y derecho) en la misma figura.
  * Tip: Usen ax.twinx() en Matplotlib para que las escalas no se aplasten mutuamente.
  * Identifiquen visualmente los momentos de Score Alto (picos) y Score Bajo (valles).

**Hipótesis a verificar:** "Durante los desplomes de mercado (Crashes), la correlación entre activos tiende a volverse perfecta (pánico generalizado), lo que colapsaría los ciclos topológicos. Por lo tanto, esperamos ver una caída drástica en el Score justo cuando el mercado se rompe, mientras que los Scores altos deberían corresponder a periodos de tendencia saludable o desacople."

**¡Manos a la obra!**